# Task axis vs. communication subspace: all ordered area pairs, one session

Is the alignment between an area's **task (context) axis** and its **communication
subspace** with another area a frontal-cortex speciality, or is it universal?

This notebook is a thin wrapper around two modules in `code/`:

- `task_axis_comm_subspace.py` -- the analysis (`analyse_session`). Full description in
  its module docstring. In short: quiescent-window firing rates, z-scored, previous-trial stimulus / response /
  reward regressed out (`NUISANCE_REGRESSORS`, `none` to skip) and **no condition-mean
  subtraction**; QC-pass single units; areas with at
  least `MIN_UNITS` (30) units, every area subsampled to exactly 30 units,
  `N_SUBSAMPLES` (10) times; LDA task axis per area with leave-one-block-out decoding
  accuracy; reduced-rank-regression communication subspace per ordered pair with 10-fold
  CV R² vs. rank and the 1-SEM dimensionality rule; alignment = norm of the unit task
  axis projected onto the subspace, tested against a dimensionality-matched random-axis
  null (the only alignment null), FDR-corrected over pairs.
- `make_alignment_figures.py` -- Figure 1 (task axis and communication subspace are
  meaningful), Figure 2 (alignment across all pairs vs. chance) per session, and pooled
  Figures 1 and 2 over every session that has results.

Reads `DATACUBE_ROOT`, `SESSION_ID` and `RESULTS_DIR` from the environment like the
other notebooks, so `run_capsule.py` runs it once per session; the pooled figure is
rebuilt at the end of every session run from all results present so far.

In [ ]:
import os, pickle, sys
from pathlib import Path

CODE_DIR = Path.cwd().parent
sys.path.insert(0, str(CODE_DIR))
import task_axis_comm_subspace as tac
import make_alignment_figures as maf

capsule_root = Path('/root/capsule/data/dynamicrouting_datacube')
root = Path(os.environ.get('DATACUBE_ROOT', capsule_root if capsule_root.exists() else CODE_DIR.parent / 'data' / 'dynamicrouting_datacube'))
session_id = os.environ.get('SESSION_ID', '743199_2024-12-05')
results_dir = Path(os.environ.get('RESULTS_DIR', '/root/capsule/results' if capsule_root.exists() else CODE_DIR.parent / 'results'))
out_dir = results_dir / session_id
out_dir.mkdir(parents=True, exist_ok=True)
print(f'DATACUBE_ROOT = {root}\nSESSION_ID = {session_id}\nresults -> {out_dir}')

# Analysis (about 6 minutes per session on a laptop)

In [ ]:
trials, units = tac.load_session(root, session_id)
res = tac.analyse_session(trials, units, session_id)
with open(out_dir / 'alignment_results.pkl', 'wb') as f:
    pickle.dump(res, f)
res['area_table'].to_csv(out_dir / 'task_axis_by_area.csv', index=False)
res['pair_table'].to_csv(out_dir / 'alignment_by_pair.csv', index=False)

# Task axis per area

In [ ]:
res['area_table'].set_index('area').round(3)

# Communication subspace and alignment per ordered pair

In [ ]:
cols = ['pair_type', 'r2', 'r2_q', 'dim', 'cos', 'cos_chance', 'cos_z_random', 'cos_q_random',
        'cos_tgt', 'cos_tgt_q_random', 'angle_deg', 'angle_chance_deg']
res['pair_table'].set_index(['source', 'target'])[cols].round(3)

In [ ]:
res['pair_table'].groupby('pair_type')[['r2', 'dim', 'cos', 'cos_chance', 'cos_z_random', 'cos_tgt']].mean().round(3)

# Figures

Saved as SVG under `<results>/<session>/figures/`; the pooled figure under `<results>/pooled/`.

In [ ]:
from IPython.display import SVG, display

fig_dir = out_dir / 'figures'
maf.figure1(res, fig_dir)
maf.figure2(res, fig_dir)
display(SVG(str(fig_dir / f'figure1_{session_id}.svg')))
display(SVG(str(fig_dir / f'figure2_{session_id}.svg')))

all_results = []
for p in sorted(results_dir.glob('*/alignment_results.pkl')):
    with open(p, 'rb') as f:
        all_results.append(pickle.load(f))
if len(all_results) > 1:
    pooled_dir = results_dir / 'pooled'
    pooled_dir.mkdir(parents=True, exist_ok=True)
    maf.pooled_figures(all_results, pooled_dir)
    display(SVG(str(pooled_dir / 'figure1_pooled.svg')))
    display(SVG(str(pooled_dir / 'figure2_pooled.svg')))
    print(f'pooled over {len(all_results)} sessions')